# Day 07 · Lazy learners and probabilistic learners: KNN & Naive Bayes

**Goal:** understand and use two very different classifiers. **K-Nearest Neighbours** (no training; distance + vote; why scaling and dimensionality matter) and **Naive Bayes** (Bayes' theorem + counting; Gaussian for numbers, Multinomial for text). You will finish with a real **SMS spam filter**.

**Run in Google Colab:** File → Open notebook → Upload → pick this `.ipynb`.

**Runtime:** CPU is fine. Nothing here takes more than a few seconds.

| Section | What you'll do |
|---|---|
| 1 | KNN from scratch in 10 lines |
| 2 | KNN in sklearn: scaling, choosing k by CV, decision regions |
| 3 | The curse of dimensionality, measured |
| 4 | KNN on handwritten digits (and looking at its mistakes) |
| 5 | Bayes' theorem by hand, then Gaussian Naive Bayes |
| 6 | A real spam filter: CountVectorizer + MultinomialNB |
| 7 | 🎯 Try it yourself |
| 8 | Homework starter + key takeaways |

## 0. Setup

Everything is preinstalled on Colab.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn
from collections import Counter
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

SEED = 42
rng = np.random.default_rng(SEED)
plt.rcParams["figure.figsize"] = (8, 4.5)
print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

## 1. KNN from scratch 🏘️

**What:** the whole algorithm in one function: distance to every stored point → k smallest → majority vote.

**Why write it by hand:** so you *feel* that there is no training step, and that every prediction re-scans the data. We use Iris (150 flowers, 4 measurements, 3 species) because you can check answers by eye.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)

def knn_predict(X_train, y_train, x_query, k=5):
    d = np.sqrt(((X_train - x_query) ** 2).sum(axis=1))   # Euclidean distance to EVERY training point
    nearest = np.argsort(d)[:k]                            # indices of the k closest
    votes = Counter(y_train[nearest])                      # count their labels
    return votes.most_common(1)[0][0]                      # majority wins

sc = StandardScaler().fit(X_tr)                            # scale on train only
Xs_tr, Xs_te = sc.transform(X_tr), sc.transform(X_te)

preds = np.array([knn_predict(Xs_tr, y_tr, xq, k=5) for xq in Xs_te])
print("scratch KNN (k=5) test accuracy:", round(accuracy_score(y_te, preds), 3))

# look at one decision in detail
q = Xs_te[0]
d = np.sqrt(((Xs_tr - q) ** 2).sum(axis=1)); idx = np.argsort(d)[:5]
print("\nquery true label:", iris.target_names[y_te[0]])
print("5 nearest neighbours:", [iris.target_names[i] for i in y_tr[idx]], "at distances", d[idx].round(2))

## 2. KNN in scikit-learn: scaling and choosing k 🎚️

Two rules you must never break with KNN:

1. **Scale features inside the pipeline.** Distance mixes features; a feature measured in hundreds drowns one measured in units.
2. **Choose k by cross-validation on the training set**, never on the test set. Small k = jagged, high variance; large k = smooth, high bias.

Watch what happens when we add one *useless* feature on a huge scale and forget to scale: the distance becomes pure noise.

In [ ]:
# Rule 1 demo: add a USELESS feature on a huge scale (think: a random "customer id" in the thousands)
noise_tr = rng.uniform(0, 10000, size=(len(X_tr), 1))
noise_te = rng.uniform(0, 10000, size=(len(X_te), 1))
X_big_tr, X_big_te = np.c_[X_tr, noise_tr], np.c_[X_te, noise_te]

unscaled = KNeighborsClassifier(n_neighbors=5).fit(X_big_tr, y_tr)
scaled   = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(X_big_tr, y_tr)
print("with a 0–10,000 noise feature — unscaled KNN:", round(unscaled.score(X_big_te, y_te), 3), "| scaled KNN:", round(scaled.score(X_big_te, y_te), 3))
print("(unscaled: distances are ~100% noise, accuracy collapses to chance; scaled: noise is 1 of 5 equal voices, the 4 real features still win)")

In [ ]:
# Rule 2: pick k with 5-fold cross-validation on the training data
ks = list(range(1, 52, 2))                                  # odd k avoids ties
cv_acc = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)), X_tr, y_tr, cv=5).mean() for k in ks]
best_k = ks[int(np.argmax(cv_acc))]

plt.plot(ks, cv_acc, marker="o"); plt.axvline(best_k, ls="--", c="#FFB43A")
plt.xlabel("k"); plt.ylabel("5-fold CV accuracy"); plt.title(f"Best k by CV = {best_k}. Tiny k overfits, huge k underfits (everything → majority class)")
plt.show()

final = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k, weights="distance")).fit(X_tr, y_tr)
print(f"test accuracy with k={best_k}, distance-weighted votes: {final.score(X_te, y_te):.3f}")

In [ ]:
# Decision regions for three values of k on two features (petal length, petal width)
from sklearn.inspection import DecisionBoundaryDisplay
X2_tr = X_tr[:, 2:4]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, k in zip(axes, [1, 15, 75]):
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X2_tr, y_tr)
    DecisionBoundaryDisplay.from_estimator(m, X2_tr, ax=ax, alpha=.35, cmap="viridis", response_method="predict")
    ax.scatter(X2_tr[:, 0], X2_tr[:, 1], c=y_tr, cmap="viridis", edgecolor="k", s=20)
    ax.set_title(f"k = {k}: {'jagged, memorises noise' if k == 1 else ('smooth, sensible' if k == 15 else 'too smooth, majority swallows a class')}")
    ax.set_xlabel("petal length"); ax.set_ylabel("petal width")
plt.tight_layout(); plt.show()

## 3. The curse of dimensionality, measured 📏

In high dimensions, random points are all roughly the *same* distance from each other: the nearest neighbour is barely nearer than the farthest. "Nearest" stops meaning anything, and KNN (and any distance-based method) degrades.

We simulate it: 1,000 random points in $d$ dimensions, and look at $\dfrac{d_{max} - d_{min}}{d_{min}}$ for a query point. It should collapse toward 0 as $d$ grows.

In [ ]:
dims = [1, 2, 3, 5, 10, 20, 50, 100, 200, 500, 1000]
contrast = []
for d in dims:
    P = rng.uniform(size=(1000, d)); q = rng.uniform(size=d)
    dist = np.sqrt(((P - q) ** 2).sum(axis=1))
    contrast.append((dist.max() - dist.min()) / dist.min())

plt.semilogx(dims, contrast, marker="o", c="#FF6B8A")
plt.xlabel("dimensions d"); plt.ylabel("(d_max − d_min) / d_min"); plt.title("Distance contrast collapses with dimension: everyone is equally far")
plt.show()
print(pd.Series(contrast, index=dims).round(2).rename("contrast"))

## 4. KNN on handwritten digits ✍️

1,797 images of 8×8 pixels = 64 features. KNN does surprisingly well here because pixel-space neighbours of a "3" really do look like 3s. We also look at *which* digits it confuses; a good habit for any classifier.

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(digits.data, digits.target, test_size=0.25, random_state=SEED, stratify=digits.target)

knn_d = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)).fit(Xd_tr, yd_tr)
pred_d = knn_d.predict(Xd_te)
print("digits test accuracy (k=3):", round(accuracy_score(yd_te, pred_d), 3))

cm = confusion_matrix(yd_te, pred_d)
print("\nconfusion matrix (rows = true digit, cols = predicted):\n", pd.DataFrame(cm))

# show the first 8 mistakes next to their nearest training neighbour
wrong = np.where(pred_d != yd_te)[0][:8]
nn_idx = knn_d[-1].kneighbors(knn_d[0].transform(Xd_te[wrong]), n_neighbors=1, return_distance=False).ravel()
fig, axes = plt.subplots(2, len(wrong), figsize=(1.6 * len(wrong), 3.6))
for j, (i, n) in enumerate(zip(wrong, nn_idx)):
    axes[0, j].imshow(Xd_te[i].reshape(8, 8), cmap="gray_r"); axes[0, j].set_title(f"true {yd_te[i]} → {pred_d[i]}", fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow(Xd_tr[n].reshape(8, 8), cmap="gray_r"); axes[1, j].set_title(f"neighbour: {yd_tr[n]}", fontsize=9); axes[1, j].axis("off")
plt.suptitle("Top: misclassified test digits. Bottom: the training digit KNN found nearest."); plt.tight_layout(); plt.show()

## 5. Bayes' theorem by hand, then Gaussian Naive Bayes 🕵️

$$P(y \mid x) = \frac{P(x \mid y)\,P(y)}{P(x)} \quad\Rightarrow\quad \hat y = \arg\max_y \; P(y)\prod_i P(x_i \mid y)$$

* **Prior** $P(y)$: class frequency.
* **Likelihood** $P(x_i \mid y)$: for numeric features, assume a Gaussian per feature per class (mean + variance = the entire model).
* **Naive**: multiply per-feature likelihoods as if features were independent.

First a 2-line worked example (a medical test), then `GaussianNB` on Iris, then we *rebuild* its `predict_proba` from its stored means and variances to prove there is nothing hidden.

In [ ]:
# Worked example: a disease affects 1% of people. Test sensitivity 95%, false-positive rate 5%. You test positive. P(disease | +)?
prior, sens, fpr = 0.01, 0.95, 0.05
p_pos = sens * prior + fpr * (1 - prior)                 # P(+) = P(+|D)P(D) + P(+|¬D)P(¬D)
posterior = sens * prior / p_pos
print(f"P(disease | positive test) = {posterior:.3f}   <- surprisingly low: the prior dominates when the disease is rare")

In [ ]:
gnb = GaussianNB().fit(X_tr, y_tr)                      # no scaling needed
print("GaussianNB test accuracy:", round(gnb.score(X_te, y_te), 3))
print("priors P(y):", gnb.class_prior_.round(3))
print("means μ (class × feature):\n", gnb.theta_.round(2))
print("variances σ² (class × feature):\n", gnb.var_.round(3))

# Rebuild predict_proba for one flower from those numbers (log-space to avoid underflow)
x = X_te[0]
log_post = np.log(gnb.class_prior_) + (-0.5 * np.log(2 * np.pi * gnb.var_) - (x - gnb.theta_) ** 2 / (2 * gnb.var_)).sum(axis=1)
post = np.exp(log_post - log_post.max()); post /= post.sum()
print("\nours   :", post.round(4))
print("sklearn:", gnb.predict_proba(x[None])[0].round(4), "| match:", np.allclose(post, gnb.predict_proba(x[None])[0]))

## 6. A real spam filter: CountVectorizer + MultinomialNB 📱

**Dataset:** the SMS Spam Collection (5,574 real messages, 13% spam). We download it from a stable GitHub mirror; if you are offline, a small built-in corpus is used instead so every later cell still runs.

**Pipeline:** `CountVectorizer` turns each message into word counts (a sparse vector over the vocabulary) → `MultinomialNB` learns $P(w \mid \text{spam})$ and $P(w \mid \text{ham})$ by counting, with **Laplace smoothing** `alpha=1.0` so an unseen word never zeroes the product.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
try:
    sms = pd.read_csv(URL, sep="\t", header=None, names=["label", "text"])
    print("downloaded SMS Spam Collection:", sms.shape)
except Exception as e:
    print("download failed, using a tiny built-in corpus:", e)
    spam = ["Congratulations you won a free prize claim now", "URGENT your account is blocked complete KYC click link", "Free loan offer approved click now",
            "Win cash lottery winner claim prize today", "Your KYC is pending urgent click link now", "Exclusive offer free recharge click here",
            "You have won lottery cash prize claim", "Loan approved zero interest click now", "Free free free win prize", "Urgent account blocked verify now"]
    ham = ["bro are we meeting at the chai stall at 5", "mom says come for dinner tomorrow", "class is cancelled tomorrow morning",
           "did you finish the ml assignment", "call me when you reach home", "happy birthday da have a great day", "send me the notes from today",
           "are you coming to cricket practice", "otp shared with you for login do not share", "lunch at the canteen at 1?"]
    sms = pd.DataFrame({"label": ["spam"] * len(spam) + ["ham"] * len(ham), "text": spam + ham})

sms["y"] = (sms.label == "spam").astype(int)
print(sms.label.value_counts(), "\n")
print(sms.sample(4, random_state=SEED)[["label", "text"]].to_string(index=False))
t_tr, t_te, ys_tr, ys_te = train_test_split(sms.text, sms.y, test_size=0.25, random_state=SEED, stratify=sms.y)

In [ ]:
spam_clf = make_pipeline(CountVectorizer(lowercase=True), MultinomialNB(alpha=1.0)).fit(t_tr, ys_tr)
pred_s = spam_clf.predict(t_te)
print("vocabulary size:", len(spam_clf[0].vocabulary_))
print("test accuracy  :", round(accuracy_score(ys_te, pred_s), 4))
print("\nconfusion matrix [[TN FP],[FN TP]]:\n", confusion_matrix(ys_te, pred_s))
print("\n", classification_report(ys_te, pred_s, target_names=["ham", "spam"], zero_division=0))
print("Baseline 'always ham' accuracy would be:", (1 - ys_te.mean()).round(3), "-> look at spam recall/precision, not accuracy.")

In [ ]:
# Which words carry the most evidence? log P(w|spam) − log P(w|ham)
vec, nb = spam_clf[0], spam_clf[1]
words = np.array(vec.get_feature_names_out())
log_odds = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
order = np.argsort(log_odds)
print("spammiest words:", list(words[order[-15:]][::-1]))
print("hammiest words :", list(words[order[:15]]))

tests = ["URGENT! Your KYC is pending, click the link to avoid account block",
         "bro are we meeting at the chai stall at 5?",
         "Congratulations! You have won a free cruise, call now to claim"]
for msg, p in zip(tests, spam_clf.predict_proba(tests)[:, 1]):
    print(f"P(spam)={p:.3f}  {msg}")

> 🧠 **Interview Q:** *"Why does Naive Bayes output probabilities like 0.9999 even when it's wrong?"* — Because it multiplies likelihoods as if words were independent; correlated words ("click", "link", "now") get counted as separate evidence, so confidence inflates. The **ranking** is usually right, the **calibration** is not. If you need honest probabilities, use logistic regression (Day 06) or calibrate (Day 12).

## 7. 🎯 Try it yourself: smoothing and the zero-probability bug (10 min)

1. Refit the spam pipeline with `MultinomialNB(alpha=1e-10)` (almost no smoothing) and `alpha=1.0` and `alpha=10.0`.
2. For each, print test accuracy and `P(spam)` for the message `"Bengaluru KYC urgent"` (the word *Bengaluru* is not in the training vocabulary; what happens to it?).
3. Which alpha would you ship? Why?

In [ ]:
# YOUR CODE HERE
msg = ["Bengaluru KYC urgent"]
for a in [1e-10, 1.0, 10.0]:
    # build make_pipeline(CountVectorizer(), MultinomialNB(alpha=a)), fit on t_tr/ys_tr
    # print a, test accuracy, predict_proba(msg)[0, 1]
    pass

<details><summary>Solution</summary>

```python
for a in [1e-10, 1.0, 10.0]:
    m = make_pipeline(CountVectorizer(), MultinomialNB(alpha=a)).fit(t_tr, ys_tr)
    print(f"alpha={a:<6} acc={m.score(t_te, ys_te):.4f}  P(spam | 'Bengaluru KYC urgent')={m.predict_proba(msg)[0,1]:.3f}")
```
"Bengaluru" is silently dropped by `CountVectorizer` at transform time (it is not in the vocabulary), so it contributes nothing. The danger of tiny alpha is different: a word seen only in ham gets P(w|spam) ≈ 0, so one such word vetoes spam no matter what else is present. alpha = 1.0 (default) is a sensible ship; tune it with CV if you care about the last 0.5%.
</details>

## Homework starter 🧪

**Easy:** paste 20 of your own SMS below (anonymised) and see what the model says. **Medium:** digits k-sweep + mistake gallery (Section 4 is your template). **Stretch:** deploy as a Gradio app on Hugging Face Spaces.

> 💼 **Career note:** a live demo link on your resume ("try my spam filter") gets clicked. Gradio + HF Spaces is free and takes 10 minutes.

In [ ]:
# --- Easy: your own messages ---
my_sms = [
    # "Dear customer, your electricity bill of Rs 2340 is pending. Pay now to avoid disconnection: bit.ly/xyz",
    # "Bhai, kal 10 baje library aa ja",
]
if my_sms:
    for msg, p in zip(my_sms, spam_clf.predict_proba(my_sms)[:, 1]):
        print(f"{'SPAM' if p >= .5 else 'ham '}  P={p:.3f}  {msg}")
else:
    print("Add a few messages to my_sms and re-run.")

# --- Stretch: a Gradio app (uncomment to run in Colab; opens a public link) ---
# !pip install -q gradio
# import gradio as gr
# def classify(text):
#     p = spam_clf.predict_proba([text])[0, 1]
#     return {"spam": float(p), "ham": float(1 - p)}
# gr.Interface(classify, gr.Textbox(lines=3, label="SMS"), gr.Label(num_top_classes=2), title="SMS Spam Filter (Naive Bayes)").launch(share=True)

## Key takeaways ✅

* **KNN**: no training; predict by majority vote of the k nearest **scaled** points. k is a bias-variance knob chosen by CV. Prediction is O(n·d) and needs the whole dataset in memory.
* **Curse of dimensionality**: in high d, all distances look alike; KNN needs few good features (or PCA first).
* **Bayes' theorem** flips conditionals: $P(y|x) \propto P(x|y)P(y)$. **Naive** = assume features independent given the class, so likelihoods are just counts / per-feature Gaussians.
* **Multinomial NB + CountVectorizer** is the classic spam filter: one pass to train, ~98% accuracy on SMS, and `alpha` (Laplace smoothing) prevents the zero-probability bug.
* NB probabilities are **overconfident**; rank with them, don't trust them.
* On imbalanced data, read **recall / precision** of the rare class, not accuracy.

**Tomorrow (Day 08):** Decision trees: a model that asks yes/no questions, ignores useless features, and explains itself.